## Creates support tables

Tables created in this script
- patterns_meta
- verb_phrase_matches
- verb_matches
- pattern_support

In [1]:
import sqlite3
import pandas as pd
import sys
sys.path.append("..")
from common_display import display_db_table 

### Configuration

In [20]:
DB_DIR = "../example_data"

PATTERN_DB = f"{DB_DIR}/verb_patterns.db"
TRANSACTION_DB = f"{DB_DIR}/transactions.db"

PATTERNS_TABLE = "patterns"
SEMANTIC_ANNOTATIONS = "semantic_annotations"
VERB_MATCHES_TABLE = "verb_matches"
VERB_PHRASE_MATCHES_TABLE = "verb_phrase_matches"
PATTERNS_META_TABLE = "patterns_meta"
PATTERN_SUPPORT_TABLE = "pattern_support"

## Connect to db

In [18]:
con = sqlite3.connect(PATTERN_DB)
cur = con.cursor()
cur.execute(f'ATTACH DATABASE "{TRANSACTION_DB}" as trans ')

## Workflow

### Abitabel asjade kättesaamiseks

Mitte-elegantne viis saada kätte kõik **head_id**-d, millele vastavates fraasides on esindatud kõik vaadeldavate mustrite osised (sobiv kääne (kui on), kaassõna (kui on), infiniitverb (kui on)). Saab kasutada ülejäänud tabelite koostamiseks. Ilmselt on võimalik teha tegelikult ära ka JOIN-ide abil. 

In [6]:
%%time

cur.execute("""DROP TABLE IF EXISTS verb_phrase_matches_step1""")
cur.execute("""DROP TABLE IF EXISTS verb_phrase_matches_step2""")
cur.execute("""DROP TABLE IF EXISTS verb_phrase_matches_step3""")

cur.execute("""
CREATE TABLE verb_phrase_matches_step1 AS
SELECT pat_id,
    tr.head_id,
    phrase_case,
    adp,
    inf_verb,
    phrase_nr
FROM 
(
    SELECT pat.pat_id as pat_id,
        tr_head.id as head_id,
        pat.phrase_case as phrase_case,
        pat.adp as adp,
        pat.inf_verb as inf_verb,
        pat.phrase_nr as phrase_nr
    FROM 
        {tbl} as pat
    INNER JOIN 
        trans.transaction_head as tr_head
    ON
        pat.verb_word=tr_head.verb
    WHERE
        pat.verb_compound=tr_head.verb_compound
) as pat_tr_joined
INNER JOIN 
    trans.`transaction` as tr
ON
    pat_tr_joined.head_id=tr.head_id
WHERE 
    pat_tr_joined.phrase_case = '' OR instr(tr.feats, pat_tr_joined.phrase_case) > 0
""".format(tbl=PATTERNS_TABLE))


cur.execute("""
CREATE TABLE verb_phrase_matches_step2 AS
SELECT 
    pat_id,
    tr.head_id,
    phrase_case,
    adp,
    inf_verb,
    phrase_nr
FROM 
    verb_phrase_matches_step1 as step1
INNER JOIN 
    trans.`transaction` as tr
ON 
    step1.head_id=tr.head_id
WHERE 
    step1.adp = '' OR (tr.form = step1.adp AND tr.deprel = 'case')
""")


cur.execute("""
CREATE TABLE verb_phrase_matches_step3 AS
SELECT pat_id,
    tr.head_id,
    phrase_case,
    adp,
    inf_verb, 
    phrase_nr
FROM 
    verb_phrase_matches_step2 as step2
INNER JOIN
    trans.`transaction` as tr
ON 
    step2.head_id=tr.head_id
WHERE 
    step2.inf_verb = '' 
    OR (tr.form = step2.inf_verb AND (instr(tr.feats, 'inf') > 0 
    OR instr(tr.feats, 'sup') > 0))
""")


cur.execute("""DROP TABLE IF EXISTS verb_phrase_matches_step1""")
cur.execute("""DROP TABLE IF EXISTS verb_phrase_matches_step2""")

CPU times: user 15.4 ms, sys: 5.35 ms, total: 20.7 ms
Wall time: 37.1 ms


### II tabel patterns_meta

In [7]:
%%time

cur.execute("""DROP TABLE IF EXISTS {tbl}""".format(tbl=PATTERNS_META_TABLE))

cur.execute("""
CREATE TABLE {tbl} (
    pat_id INTEGER,
    phrase_count INTEGER
)
""".format(tbl=PATTERNS_META_TABLE))

cur.execute("""
INSERT INTO {tbl}(
    pat_id,
    phrase_count
)
SELECT 
    pat_id,
    count(*) AS phrase_count
FROM
(
    SELECT DISTINCT
        pat_id, 
        head_id
    FROM
        verb_phrase_matches_step3
) AS tbl
GROUP BY
    tbl.pat_id
ORDER BY
    phrase_count DESC
""".format(tbl=PATTERNS_META_TABLE))

cur.execute("""CREATE INDEX meta_pat_id_idx ON {tbl}(pat_id)""".format(tbl=PATTERNS_META_TABLE))

con.commit()

CPU times: user 1.94 ms, sys: 2.47 ms, total: 4.41 ms
Wall time: 11.8 ms


### III tabel verb_phrase_matches

In [8]:
%%time

cur.execute("""
DROP TABLE IF EXISTS {tbl}
""".format(tbl=VERB_PHRASE_MATCHES_TABLE))

cur.execute("""
CREATE TABLE {tbl} (
    pat_id INTEGER,
    head_id INTEGER,
    phrase_nr INTEGER
)
""".format(tbl=VERB_PHRASE_MATCHES_TABLE))

cur.execute("""
INSERT INTO {tbl}(
    pat_id,
    head_id,
    phrase_nr
)
SELECT DISTINCT
    pat_id,
    head_id,
    phrase_nr
FROM
    verb_phrase_matches_step3
""".format(tbl=VERB_PHRASE_MATCHES_TABLE))

cur.execute("""
CREATE INDEX match_pat_id_idx ON {tbl}(pat_id)
""".format(tbl=VERB_PHRASE_MATCHES_TABLE))

cur.execute("""
CREATE INDEX match_head_id_idx ON {tbl}(head_id)
""".format(tbl=VERB_PHRASE_MATCHES_TABLE))

cur.execute("""
CREATE INDEX match_phrase_nr_idx ON {tbl}(phrase_nr)
""".format(tbl=VERB_PHRASE_MATCHES_TABLE))


con.commit()

CPU times: user 2.86 ms, sys: 2.43 ms, total: 5.29 ms
Wall time: 16.7 ms


### IV tabel verb_matches

In [24]:
%%time

cur.execute("""DROP TABLE IF EXISTS {tbl}""".format(tbl=VERB_MATCHES_TABLE))

cur.execute("""DROP TABLE IF EXISTS verb_matches_1""")

cur.execute("""
CREATE TABLE verb_matches_1 (
    pat_id INTEGER,
    head_id INTEGER
)
""")

cur.execute("""
INSERT INTO verb_matches_1 (
    pat_id,
    head_id
)
SELECT DISTINCT
    pat_id,
    head_id
FROM
(
    SELECT pat_id, tr_head.id as head_id
    FROM
        {tbl} as pat
    INNER JOIN
        trans.transaction_head as tr_head
    ON
        pat.verb_word = tr_head.verb
    WHERE pat.verb_compound = tr_head.verb_compound
)
""".format(tbl=PATTERNS_TABLE))

cur.execute(
"""
CREATE TABLE {tbl1} AS 
Select distinct
    vm.pat_id,
    vm.head_id,
    sem.phrase_nr,
    sem.semantic_role,
    sem.certainty
from 
verb_matches_1 as vm
join 
{tbl2} as sem
on vm.pat_id = sem.pattern_id 
""".format(tbl1=VERB_MATCHES_TABLE, tbl2=SEMANTIC_ANNOTATIONS)
)

cur.execute("""DROP TABLE IF EXISTS verb_matches_1""")

cur.execute("""CREATE INDEX v_match_pat_id_idx ON {tbl}(pat_id)""".format(tbl=VERB_MATCHES_TABLE))
cur.execute("""CREATE INDEX v_match_head_id_idx ON {tbl}(head_id)""".format(tbl=VERB_MATCHES_TABLE))

con.commit()

CPU times: user 42.7 ms, sys: 6.86 ms, total: 49.5 ms
Wall time: 52.6 ms


soovi korral saab kustutada abitabeli, aga SQLITE-s see tegevus mäluruumi ei vabasta

In [25]:
cur.execute("""DROP TABLE IF EXISTS verb_phrase_matches_step3""")

### V tabel pattern_support

In [ ]:
%%time

cur.execute("""DROP TABLE IF EXISTS {tbl}""".format(tbl=PATTERN_SUPPORT_TABLE))

cur.execute("""
CREATE TABLE {new_table} AS
SELECT
    pm.pat_id,
    verb_word,
    verb_compound,
    phrase_case,
    adp,
    inf_verb,
    verb_match_count AS verb_occurrence_count,
    phrase_count AS absolute_support,
    CAST(phrase_count AS REAL) / CAST(verb_match_count AS REAL) * 100 AS relative_support
FROM
(
    SELECT 
        pat.pat_id as pat_id,
        verb_word,
        verb_compound,
        phrase_case,
        adp,
        inf_verb,
        count(*) AS verb_match_count
    FROM
        {tbl1} as vm
    INNER JOIN
        {tbl2} as pat
    ON
        pat.pat_id = vm.pat_id
    GROUP BY
        pat.pat_id
) as tbl
INNER JOIN
    {tbl3} as pm
ON
    tbl.pat_id = pm.pat_id
ORDER BY
    relative_support DESC
""".format(new_table=PATTERN_SUPPORT_TABLE, tbl1=VERB_MATCHES_TABLE, tbl2=PATTERNS_TABLE, tbl3=PATTERNS_META_TABLE))

## Check db and table contents

In [27]:
display_db_table(con, VERB_MATCHES_TABLE, 10, 'head')

,pat_id,head_id,phrase_nr,semantic_role,certainty
0,1,54,1,isik,vahel
1,1,54,1,koht,vahel
2,1,54,1,muu,mitte kunagi
3,1,74,1,isik,vahel
4,1,74,1,koht,vahel
5,1,74,1,muu,mitte kunagi
6,1,96,1,isik,vahel
7,1,96,1,koht,vahel
8,1,96,1,muu,mitte kunagi
9,1,130,1,isik,vahel


In [28]:
# ühenduse sulgemine
con.close()